# 大六壬核心引擎程式化調用與數據分析指南

本指南旨在展示如何透過 Python 程式碼直接驅動 `daliuren-web-engine` 的核心算力。本專案不僅提供 Web 介面，其底層建構的排盤物件模型亦具備高度的擴充性，適合用於學術回測、統計分析或量化研究。

## 環境準備
請確保您已安裝本專案所需的基礎套件：

`pip install fastapi uvicorn opencc-python-reimplemented pydantic`

In [6]:
import os
import sys

# 修正 Python 模組搜尋路徑，確保不論在何處啟動 Notebook 都能正確載入 app 與 repo_core
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

repo_core_dir = os.path.join(current_dir, "app", "repo_core")
if repo_core_dir not in sys.path:
    sys.path.insert(0, repo_core_dir)

print("模組搜尋路徑初始化完成。")

模組搜尋路徑初始化完成。


In [9]:
from common import GetLi, GetShiChen, DiZHiList
from shipan.shipan import ShiPan

# 1. 定義時間切片並計算曆法參數
target_year, target_month, target_day = 2026, 5, 20
target_hour, target_minute, target_second = 14, 33, 0

li_data = GetLi(target_year, target_month, target_day, target_hour, target_minute, target_second)

# 2. 轉換核心物件為地支純字串
yuejiang_obj = li_data[4]
yuejiang_str = DiZHiList[yuejiang_obj.num - 1]

zhanshi_obj = GetShiChen(target_hour)
zhanshi_str = DiZHiList[zhanshi_obj.num - 1]

# 3. 直接實例化大六壬實戰盤物件 (ShiPan)
sp = ShiPan(
    year=target_year, month=target_month, day=target_day,
    hour=target_hour, minutes=target_minute, second=target_second,
    月将=yuejiang_str, 占时=zhanshi_str,
    昼占=True, 占测的事="大數據統計分析",
    性别=0, 生年=2026
)

# 4. 程式化數據提取與欄位檢視 (注意屬性需對齊源碼的簡體名稱)
print("=== 大六壬底層數據結構提取 ===")
print(f"西元時間: {sp.year}-{sp.month:02d}-{sp.day:02d} {sp.hour:02d}:{sp.minutes:02d}")
print(f"四柱八字: {sp.四柱与节气[0]}年 {sp.四柱与节气[1]}月 {sp.四柱与节气[2]}日 {sp.四柱与节气[3]}時")
print(f"目前節氣: {sp.四柱与节气[5]} ｜ 下一個中氣: {sp.四柱与节气[6]}")
print(f"當前月將: {sp.yueJiang} ｜ 占時: {sp.zhanShi}")
print(f"旬落空亡: {[str(k) for k in sp.空亡]}")
print(f"觸發卦體格局: {sp.格局}")

print("\n--- 三傳狀態 (事態軌跡) ---")
print(f"初傳 (發用): {sp.sc.初} (六親: {sp.sc.六亲[0]} / 遁干: {sp.sc.遁干[0]} / 天將: {sp.tianJiang[sp.sc.初]})")
print(f"中傳:         {sp.sc.中} (六親: {sp.sc.六亲[1]} / 遁干: {sp.sc.遁干[1]} / 天將: {sp.tianJiang[sp.sc.中]})")
print(f"末傳:         {sp.sc.末} (六親: {sp.sc.六亲[2]} / 遁干: {sp.sc.遁干[2]} / 天將: {sp.tianJiang[sp.sc.末]})")

print("\n--- 四課狀態 (主客現狀) ---")
print(f"第一課 (日干陽神): 課上 {sp.sk.干阳神} / 課下 {sp.sk.干}")
print(f"第二課 (日干陰神): 課上 {sp.sk.干阴神} / 課下 {sp.sk.干阳神}")
print(f"第三課 (日支陽神): 課上 {sp.sk.支阳神} / 課下 {sp.sk.支}")
print(f"第四課 (日支陰神): 課上 {sp.sk.支阴神} / 課下 {sp.sk.支阳神}")

print("\n--- 天地盤動態查詢 (12宮氣數) ---")
# 查詢任意地盤位置上，目前坐著什麼天盤神將
for dz in ["子", "午", "卯", "酉"]:
    from ganzhiwuxin.ganzhiwuxin import 支
    target_dz = 支(dz)
    print(f"地盤 [{dz}宮] 上臨天盤：{sp.tp[target_dz]} (乘天將: {sp.tianJiang[sp.tp[target_dz]]})")

=== 大六壬底層數據結構提取 ===
西元時間: 2026-05-20 14:33
四柱八字: 丙午年 癸巳月 甲午日 辛未時
目前節氣: 立夏 2026-05-05 19:48:42 ｜ 下一個中氣: 小滿 2026-05-21 08:36:44
當前月將: 酉 ｜ 占時: 未
旬落空亡: ['辰', '巳']
觸發卦體格局: ['涉害卦', '三陽卦', '六儀卦']

--- 三傳狀態 (事態軌跡) ---
初傳 (發用): 辰 (六親: 財 / 遁干:  / 天將: 合)
中傳:         午 (六親: 子 / 遁干: 甲 / 天將: 龙)
末傳:         申 (六親: 官 / 遁干: 丙 / 天將: 虎)

--- 四課狀態 (主客現狀) ---
第一課 (日干陽神): 課上 辰 / 課下 甲
第二課 (日干陰神): 課上 午 / 課下 辰
第三課 (日支陽神): 課上 申 / 課下 午
第四課 (日支陰神): 課上 戌 / 課下 申

--- 天地盤動態查詢 (12宮氣數) ---
地盤 [子宮] 上臨天盤：寅 (乘天將: 蛇)
地盤 [午宮] 上臨天盤：申 (乘天將: 虎)
地盤 [卯宮] 上臨天盤：巳 (乘天將: 勾)
地盤 [酉宮] 上臨天盤：亥 (乘天將: 阴)


## 物件序列化（將盤勢轉換為繁體結構化 JSON）

在進行分散式架構開發、將排盤結果存入 NoSQL 資料庫（如 MongoDB），或是將數據傳輸給外部分析平台時，需要將 `ShiPan` 物件轉換為標準的 JSON 格式。

以下步驟在不變更任何底層核心原始碼的前提下，利用 Python 內建的字典映射與 `opencc` 轉換器，將全盤氣數、三傳、四課及格局，精確導出 JSON 數據。

In [12]:
import json
from opencc import OpenCC

# 初始化繁簡轉換器，確保輸出結果為標準繁體
cc = OpenCC('s2t')

# 預先提取空亡字串清單，用於傳內比對
kw_list = [str(k) for k in sp.空亡]

# 封裝高度結構化的排盤數據字典
shipan_json_dict = {
    "基本資訊": {
        "時間": f"{sp.year}-{sp.month:02d}-{sp.day:02d} {sp.hour:02d}:{sp.minutes:02d}:{sp.second:02d}",
        "四柱八字": {
            "年柱": str(sp.四柱与节气[0]),
            "月柱": str(sp.四柱与节气[1]),
            "日柱": str(sp.四柱与节气[2]),
            "時柱": str(sp.四柱与节气[3])
        },
        "節氣": {
            "當前節氣": str(sp.四柱与节气[5]),
            "下一中氣": str(sp.四柱与节气[6])
        },
        "月將": str(sp.yueJiang),
        "占時": str(sp.zhanShi),
        "晝夜": "晝占" if sp.昼占 else "夜占",
        "空亡": kw_list
    },
    "三傳": {
        "初傳": {
            "六親": str(sp.sc.六亲[0]),
            "遁干": str(sp.sc.遁干[0]),
            "地支": str(sp.sc.初),
            "天將": str(sp.tianJiang[sp.sc.初]),
            "是否空亡": str(sp.sc.初) in kw_list
        },
        "中傳": {
            "六親": str(sp.sc.六亲[1]),
            "遁干": str(sp.sc.遁干[1]),
            "地支": str(sp.sc.中),
            "天將": str(sp.tianJiang[sp.sc.中]),
            "是否空亡": str(sp.sc.中) in kw_list
        },
        "末傳": {
            "六親": str(sp.sc.六亲[2]),
            "遁干": str(sp.sc.遁干[2]),
            "地支": str(sp.sc.末),
            "天將": str(sp.tianJiang[sp.sc.末]),
            "是否空亡": str(sp.sc.末) in kw_list
        }
    },
    "四課": {
        "第一課": { "課上": str(sp.sk.干阳神), "課下": str(sp.sk.干), "天將": str(sp.tianJiang[sp.sk.干阳神]) },
        "第二課": { "課上": str(sp.sk.干阴神), "課下": str(sp.sk.干阳神), "天將": str(sp.tianJiang[sp.sk.干阴神]) },
        "第三課": { "課上": str(sp.sk.支阳神), "課下": str(sp.sk.支), "天將": str(sp.tianJiang[sp.sk.支阳神]) },
        "第四課": { "課上": str(sp.sk.支阴神), "課下": str(sp.sk.支阳神), "天將": str(sp.tianJiang[sp.sk.支阴神]) }
    },
    "卦體格局": sp.格局
}

# 轉譯為 JSON 格式字串（禁用 ASCII 編碼以保留中文字元）
json_raw_str = json.dumps(shipan_json_dict, ensure_ascii=False, indent=2)

# 進行繁體化轉換
json_traditional_str = cc.convert(json_raw_str)

# 執行大六壬專屬經典術語校正，防止過度轉換
json_traditional_str = json_traditional_str.replace('天後', '天后')
json_traditional_str = json_traditional_str.replace('晝佔', '晝占')
json_traditional_str = json_traditional_str.replace('夜佔', '夜占')
json_traditional_str = json_traditional_str.replace('佔時', '占時')
json_traditional_str = json_traditional_str.replace('幹', '干')

# 輸出最終結構化 JSON
print(json_traditional_str)

{
  "基本資訊": {
    "時間": "2026-05-20 14:33:00",
    "四柱八字": {
      "年柱": "丙午",
      "月柱": "癸巳",
      "日柱": "甲午",
      "時柱": "辛未"
    },
    "節氣": {
      "當前節氣": "立夏 2026-05-05 19:48:42",
      "下一中氣": "小滿 2026-05-21 08:36:44"
    },
    "月將": "酉",
    "占時": "未",
    "晝夜": "晝占",
    "空亡": [
      "辰",
      "巳"
    ]
  },
  "三傳": {
    "初傳": {
      "六親": "財",
      "遁干": "",
      "地支": "辰",
      "天將": "合",
      "是否空亡": true
    },
    "中傳": {
      "六親": "子",
      "遁干": "甲",
      "地支": "午",
      "天將": "龍",
      "是否空亡": false
    },
    "末傳": {
      "六親": "官",
      "遁干": "丙",
      "地支": "申",
      "天將": "虎",
      "是否空亡": false
    }
  },
  "四課": {
    "第一課": {
      "課上": "辰",
      "課下": "甲",
      "天將": "合"
    },
    "第二課": {
      "課上": "午",
      "課下": "辰",
      "天將": "龍"
    },
    "第三課": {
      "課上": "申",
      "課下": "午",
      "天將": "虎"
    },
    "第四課": {
      "課上": "戌",
      "課下": "申",
      "天將": "玄"
    }
  },
  "卦體格局": [
    "涉害卦",
    "三陽卦",
    "六儀卦"
  